# Kaudulla Elephant Movement Prediction with Ensemble Learning

**Dataset:** `data/kaudulla_elephants_clean.csv` (Sri Lankan Kaudulla elephant GPS telemetry)  
**Task:** Predict an engineered movement-speed class from the current GPS context.  
**Models:** Random Forest (Bagging), XGBoost and LightGBM (Boosting), Stacking, and Random Forest + Artificial Neural Network (Hybrid).

> Run this notebook from the project root. Install dependencies first with `pip install -r requirements.txt`. The complete reusable training pipeline is in `train_models.py`; this notebook documents the workflow, executes it, and analyses its saved outputs.

## 1. Research question and dataset understanding

Can ensemble models classify the movement-speed category associated with a Kaudulla elephant GPS observation using identity, location, and temporal context? The supplied CSV has 86,747 records and eight original columns. Latitude and longitude are missing in many rows, so only valid consecutive GPS pairs can be used to derive movement speed.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
DATA_PATH = ROOT / 'data' / 'kaudulla_elephants_clean.csv'
raw = pd.read_csv(DATA_PATH)
print('Dataset shape:', raw.shape)
display(raw.head())
display(raw.dtypes.to_frame('data type'))
display(raw.isna().sum().to_frame('missing values'))

## 2. Preprocessing and target engineering

1. Parse UTC timestamps and retain rows with valid coordinates in plausible Sri Lankan bounds.
2. Sort by collar ID and timestamp. Pair each fix with the next fix for the same collar.
3. Retain time gaps from 0.25 to 2.5 hours and calculate great-circle distance using the Haversine formula.
4. Exclude estimated speeds above 20 km/h to reduce the effect of implausible GPS jumps.
5. Define three dataset-specific speed groups using the 33rd and 67th percentiles: **Low Movement**, **Moderate Movement**, and **High Movement**. The quantile thresholds reduce severe class imbalance in the supplied telemetry; they are operational thresholds, not validated biological labels.

In [ ]:
from train_models import build_dataset, LABELS

processed, model_data, feature_columns = build_dataset()
print('Original rows:', len(raw))
print('Rows after valid movement-pair filtering:', len(model_data))
print('Features:', feature_columns)
display(model_data['movement_class'].value_counts().rename('records').to_frame())

plt.figure(figsize=(7, 4))
model_data['movement_class'].value_counts().reindex(LABELS).plot(kind='bar')
plt.title('Engineered movement class distribution')
plt.xlabel('Movement class'); plt.ylabel('GPS pairs'); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 3. Feature preparation and experimental design

The model inputs include collar ID, elephant name, sex, current latitude/longitude, local hour, weekday, month, weekend flag, and cyclical hour/month features. The next GPS coordinates, elapsed time, and calculated speed are excluded from the predictors to avoid direct target leakage. Numeric values are median-imputed and standardised; categorical values are mode-imputed and one-hot encoded. A stratified 80/20 train-test split with random state 42 is used for this experiment. A small GridSearchCV experiment tunes Random Forest max_depth and min_samples_leaf on a reproducible training subset using 3-fold weighted F1.

In [ ]:
from sklearn.model_selection import train_test_split
from train_models import make_preprocessor

X = model_data[feature_columns]
y = pd.Series(pd.Categorical(model_data['movement_class'], categories=LABELS).codes, index=model_data.index)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
preprocessor, numeric_features, categorical_features = make_preprocessor()
print('Training rows:', len(X_train), '| Test rows:', len(X_test))
print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

## 4. Train and compare ensemble approaches

The training script fits: (1) Random Forest bagging; (2) XGBoost and LightGBM boosting; (3) a stacking classifier combining Random Forest, Extra Trees and XGBoost with logistic regression as the meta-learner; and (4) a hybrid soft-voting model combining Random Forest and a multilayer perceptron neural network. Models are evaluated on the same held-out test partition using accuracy, weighted precision, weighted recall, and weighted F1-score. The training pipeline saves the best deployment model and comparison artefacts under `models/` and `results/`.

In [ ]:
# Reproducible full model training. This can take several minutes.
from train_models import main as train_all_models
train_all_models()

## 5. Model comparison results

In [ ]:
metrics = pd.read_csv(ROOT / 'results' / 'model_metrics.csv')
display(metrics.sort_values('F1-score (weighted)', ascending=False).style.format({
    'Accuracy': '{:.3f}', 'Precision (weighted)': '{:.3f}',
    'Recall (weighted)': '{:.3f}', 'F1-score (weighted)': '{:.3f}'
}))

ax = metrics.sort_values('F1-score (weighted)').plot(
    x='Model', y='F1-score (weighted)', kind='barh', figsize=(9, 5), legend=False
)
ax.set_xlabel('Weighted F1-score'); ax.set_ylabel(''); ax.set_title('Ensemble model comparison')
ax.set_xlim(0, 1); plt.tight_layout(); plt.show()

## 6. Feature importance and error analysis

In [ ]:
importance = pd.read_csv(ROOT / 'results' / 'feature_importance.csv').head(15).sort_values('Importance')
ax = importance.plot(x='Feature', y='Importance', kind='barh', figsize=(9, 6), legend=False)
ax.set_title('Random Forest feature importance'); ax.set_xlabel('Importance'); ax.set_ylabel('')
plt.tight_layout(); plt.show()

confusion = pd.read_csv(ROOT / 'results' / 'best_confusion_matrix.csv', index_col=0)
display(confusion)
plt.figure(figsize=(6, 5))
plt.imshow(confusion.values, cmap='Greens')
plt.xticks(range(len(LABELS)), LABELS, rotation=25, ha='right')
plt.yticks(range(len(LABELS)), LABELS)
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Best model confusion matrix')
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        plt.text(j, i, int(confusion.iloc[i, j]), ha='center', va='center')
plt.colorbar(); plt.tight_layout(); plt.show()

## 7. Interpretation and research reflection

Use the generated metrics rather than assuming one model must win. Tree ensembles are generally well suited to structured tabular data because they can model nonlinear interactions between location, time, and elephant identity. Random Forest reduces variance by averaging trees; boosting methods iteratively improve earlier errors; stacking learns how to combine model outputs; and the hybrid experiment combines tree and neural-network probabilities. A neural network may learn complementary nonlinear patterns but introduces additional tuning and scaling requirements. These experiments use tabular telemetry features, not raw images or text, so they do not establish performance on image/text data.

**Important interpretation caveat:** the random split may place observations from the same collar/elephant in both training and test sets. High scores therefore do not prove generalisation to unseen elephants or another reserve. A stronger follow-up is GroupShuffleSplit or GroupKFold using collar ID as the group, plus temporal holdout evaluation. The stacking model's result should be discussed honestly even if it is weaker than other approaches.

In [ ]:
summary = json.loads((ROOT / 'results' / 'run_summary.json').read_text())
print('Best model by weighted F1:', summary['best_model'])
print('Weighted F1:', round(summary['best_f1_weighted'], 4))
print('Class distribution:', summary['class_distribution'])
print('Rows retained:', summary['rows_after_valid_movement_filter'])
print('Random Forest GridSearchCV:', summary['random_forest_grid_search'])

## 8. Conclusion

This project builds a reproducible classification pipeline on a local Sri Lankan wildlife telemetry dataset and compares Bagging, Boosting, Stacking, and a Hybrid ensemble. The Streamlit dashboard exposes the model comparison, sampled GPS observations, Random Forest feature importance, confusion matrix, methodology, and a prediction form. The derived labels and validation limitations must be explained clearly in any academic submission.

## 9. AI Usage Acknowledgment

Generative AI (ChatGPT) assisted with boilerplate code structure, dashboard layout, documentation wording, and notebook scaffolding. 